# Data collection for the year 2023

In [1]:
import cocopp
dsl = cocopp.load("bbob/2023/*")

  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2023/BIRMIN_Kudela.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2023\BIRMIN_Kudela.tgz
  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2023/DIRECT-REV_Kudela.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2023\DIRECT-REV_Kudela.tgz
  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2023/E-WOA_Espinoza.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2023\E-WOA_Espinoza.tgz
  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2023/I-DBDP-GL_Kudela.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2023\I-DBDP-GL_Kudela.tgz
  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2023/WOA_Espinoza.tgz to C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2023\WOA_Espinoza.tgz
  downloading https://numbbo.github.io/data-archive/data-archive/bbob/2023/a-CMA-ES_Gi

In [2]:
import numpy as np

dd = dsl.dictByDimFunc()     # your grouped datasets
t = 1e-8                     # choose the target precision

best_by_df = {}              # (dim, fid) -> (best_alg, best_ert)

for dim in sorted(dd.keys()): 
    for fid in sorted(dd[dim].keys()):
        rows = []
        for ds in dd[dim][fid]:                 # each ds = one algorithm
            ert = float(ds.detERT([t])[0])      # ERT in #evals at target t
            rows.append((ds.algId, ert))  
        # ignore INF (not reached) when picking best
        finite = [(a, e) for (a, e) in rows if np.isfinite(e)] 
    
        if finite:
            best_alg, best_ert = min(finite, key=lambda x: x[1]) 
        else:
            best_alg, best_ert = None, np.inf
        best_by_df[(dim, fid)] = (best_alg, best_ert) 
        print(f"dim={dim:>2}, F{fid:>2} -> {best_alg}  (ERT={best_ert:.3g} @ {t})")


dim= 2, F 1 -> DIRECT-REV_Kudela  (ERT=37.9 @ 1e-08)
dim= 2, F 2 -> BIRMIN_Kudela  (ERT=116 @ 1e-08)
dim= 2, F 3 -> DIRECT-REV_Kudela  (ERT=638 @ 1e-08)
dim= 2, F 4 -> DIRECT-REV_Kudela  (ERT=604 @ 1e-08)
dim= 2, F 5 -> default-CMA-ES_Gissler  (ERT=25 @ 1e-08)
dim= 2, F 6 -> BIRMIN_Kudela  (ERT=150 @ 1e-08)
dim= 2, F 7 -> adm-CMA-ES_Gissler  (ERT=227 @ 1e-08)
dim= 2, F 8 -> DIRECT-REV_Kudela  (ERT=103 @ 1e-08)
dim= 2, F 9 -> DIRECT-REV_Kudela  (ERT=85.2 @ 1e-08)
dim= 2, F10 -> a-CMA-ES_Gissler  (ERT=432 @ 1e-08)
dim= 2, F11 -> a-CMA-ES_Gissler  (ERT=437 @ 1e-08)
dim= 2, F12 -> adm-CMA-ES_Gissler  (ERT=952 @ 1e-08)
dim= 2, F13 -> s-CMA-ES_Gissler  (ERT=553 @ 1e-08)
dim= 2, F14 -> a-CMA-ES_Gissler  (ERT=435 @ 1e-08)
dim= 2, F15 -> DIRECT-REV_Kudela  (ERT=516 @ 1e-08)
dim= 2, F16 -> cd-CMA-ES_Gissler  (ERT=694 @ 1e-08)
dim= 2, F17 -> cma-bt-bbob_Brockhoff  (ERT=1.75e+03 @ 1e-08)
dim= 2, F18 -> a-CMA-ES_Gissler  (ERT=2.35e+03 @ 1e-08)
dim= 2, F19 -> dm-CMA-ES_Gissler  (ERT=1.44e+03 @ 1e-08

In [3]:
from collections import Counter, defaultdict

In [4]:
# Build a frequency counter: how many (dim,fid) each algo wins
win_counter = Counter(
    alg for (alg, ert) in best_by_df.values()
    if alg is not None and np.isfinite(ert)
)

# If you want a plain dict:
wins_dict = dict(win_counter)

# (Optional) pretty print, most wins first
for alg, count in win_counter.most_common():
    print(f"{alg}: {count}")

a-CMA-ES_Gissler: 23
DIRECT-REV_Kudela: 18
adm-CMA-ES_Gissler: 18
I-DBDP-GL_Kudela: 14
ad-CMA-ES_Gissler: 13
cma-bp-bbob_Brockhoff: 7
default-CMA-ES_Gissler: 6
BIRMIN_Kudela: 5
sd-CMA-ES_Gissler: 5
sdm-CMA-ES_Gissler: 5
d-CMA-ES_Gissler: 5
s-CMA-ES_Gissler: 4
dm-CMA-ES_Gissler: 4
cd-CMA-ES_Gissler: 3
cma-bt-bbob_Brockhoff: 3
c-CMA-ES_Gissler: 2
sc-CMA-ES_Gissler: 1
cdm-CMA-ES_Gissler: 1


In [5]:
"""
Given best_by_df: {(dim, fid): (alg, ert)},
return {dim: algo_with_most_(fid)_wins_in_that_dim}.
Tie-break: lower total ERT across that dim, then alphabetical.
    """
wins = defaultdict(Counter)                    # dim -> Counter({alg: count})
ert_sums = defaultdict(lambda: defaultdict(float))  # dim -> {alg: total_ert}

for (dim, fid), (alg, ert) in best_by_df.items():
    if alg is None or not np.isfinite(ert):
        continue
    wins[dim][alg] += 1
    ert_sums[dim][alg] += float(ert)

result = {}
for dim, counter in wins.items():
    max_wins = max(counter.values())
    candidates = [a for a, c in counter.items() if c == max_wins]
    best = min(candidates, key=lambda a: (ert_sums[dim][a], a))  # tie-breaks
    result[dim] = best
result


{2: 'DIRECT-REV_Kudela',
 3: 'I-DBDP-GL_Kudela',
 5: 'I-DBDP-GL_Kudela',
 10: 'adm-CMA-ES_Gissler',
 20: 'a-CMA-ES_Gissler',
 40: 'a-CMA-ES_Gissler'}

In [6]:
import numpy as np
import pandas as pd

# Make sure 'dd' already exists
# (if not, run: dsl = cocopp.load('path/to/your/ppdata'); dd = dsl.dictByDimFunc())

targets = [1e-1, 1e-2, 1e-3, 1e-5, 1e-8]
rows = []  # reset before starting the full loop

for dim in sorted(dd.keys()):                      # e.g. [2, 3, 5, 10, 20, 40]
    for fid in sorted(dd[dim].keys()):
        for t in targets:
            algo_erts = []
            for ds in dd[dim][fid]:                # each algorithm
                ert = float(ds.detERT([t])[0])
                algo_erts.append((ds.algId, ert))
            
            finite = [(a, e) for (a, e) in algo_erts if np.isfinite(e)]

            if finite:
                best_alg, best_ert = min(finite, key=lambda x: x[1])
            else:
                best_alg, best_ert = None, np.inf

            rows.append({
                "dimension": dim,
                "function_id": fid,
                "target": t,
                "best_algorithm": best_alg,
                "best_ERT": best_ert
            })

# Build DataFrame
df_best = pd.DataFrame(rows)
df_best = df_best.sort_values(by=["dimension", "function_id", "target"]).reset_index(drop=True)

# Confirm dimensions included
print("✅ Unique dimensions in table:", df_best["dimension"].unique())
print(df_best.head(15))


✅ Unique dimensions in table: [ 2  3  5 10 20 40]
    dimension  function_id        target     best_algorithm    best_ERT
0           2            1  1.000000e-08  DIRECT-REV_Kudela   37.866667
1           2            1  1.000000e-05  DIRECT-REV_Kudela   31.666667
2           2            1  1.000000e-03  DIRECT-REV_Kudela   21.666667
3           2            1  1.000000e-02  DIRECT-REV_Kudela   21.466667
4           2            1  1.000000e-01  DIRECT-REV_Kudela   19.200000
5           2            2  1.000000e-08      BIRMIN_Kudela  116.466667
6           2            2  1.000000e-05  DIRECT-REV_Kudela   75.000000
7           2            2  1.000000e-03  DIRECT-REV_Kudela   62.800000
8           2            2  1.000000e-02  DIRECT-REV_Kudela   53.533333
9           2            2  1.000000e-01  DIRECT-REV_Kudela   42.466667
10          2            3  1.000000e-08  DIRECT-REV_Kudela  637.600000
11          2            3  1.000000e-05  DIRECT-REV_Kudela  628.533333
12          2 

In [7]:
import os
os.makedirs("results", exist_ok=True)

df_best.to_csv("results/best_algos_2023.csv", index=False)
